In [ ]:
import csv
from collections import Counter, defaultdict
from pathlib import Path
from urllib.request import urlopen

import marimo as mo
import plotly.graph_objects as go

# Protein fitness jumps when mutations are composed, not discovered alone

> **Competition dataset used: ProteinGym / FLIP2 protein fitness landscape benchmark family.**
> In a Novartis imine reductase (IRED) engineering campaign, the single-mutant landscape looks almost barren:
> across **11,305 screened variants**, the median activity ratio is only **0.006**.
> But the campaign's later combinatorial rounds expose the real signal: **3-mutation variants reach
> 0.626 median activity**, and **EPPCR3 variants pair 98.2% median R-selectivity with 0.70 median activity**.
>
> The practical insight: for this enzyme, the winning designs are not isolated "best mutations";
> they are reusable mutation programs, especially variants built around **S220T**, often with **H230Y**.
>
> **The killer stat: 34 / 34 high-fitness winners (activity ≥ 0.70, R-ee ≥ 95%) carry S220T —
> vs. 78 / 427 (18.3%) of the full selectivity table.** A 5.5× enrichment. The motif is the signal;
> the screen is the noise.

## The data

This notebook uses the IRED protein fitness dataset bundled with the PyData hackathon starter repo
(a Novartis machine-directed evolution campaign for an imine reductase, originally published as
*Machine-Directed Evolution of an Imine Reductase for Activity and Stereoselectivity*).

The data is not literally a download from FLIP2 or ProteinGym, but it is exactly the class of data
those benchmarks formalize — variant-level activity and selectivity assays on a single protein with
explicit mutation strings — so I analyze it as the kind of **protein fitness landscape** problem the
**FLIP2 / ProteinGym** benchmark families were built around. Each row is a protein variant, each
mutation string describes a sequence change, and the assay columns measure fitness-like outcomes:

- `ratio`: normalized activity/product-formation signal; higher is better.
- `r_enantiomeric_excess`: stereoselectivity toward the desired R product; higher is better.
- `experiment`: engineering round or selection strategy.

The notebook asks one question: **when did the campaign stop finding merely active variants and start
finding variants that are both active and selective?**

In [ ]:
DATA_DIR = Path(".cache") / "ired-novartis"
SCREEN_PATH = DATA_DIR / "cs1c02786_si_002.csv"
SELECTIVITY_PATH = DATA_DIR / "cs1c02786_si_003.csv"
RAW_BASE_URL = (
    "https://raw.githubusercontent.com/ericmjl/2026-pydata-boston-cursor-hackathon"
    "/main/data/ired-novartis/"
)

def open_text(path):
    if path.exists():
        return path.open(newline="")
    return (line.decode("utf-8") for line in urlopen(RAW_BASE_URL + path.name))

def mutation_count(mutation):
    return len([part for part in mutation.split(";") if part.strip()])

def load_screen():
    out = []
    with open_text(SCREEN_PATH) as f:
        for screen_row in csv.DictReader(f):
            screen_row["ratio"] = float(screen_row["ratio"])
            screen_row["mean"] = float(screen_row["mean"])
            screen_row["alpha"] = float(screen_row["alpha"])
            screen_row["count"] = int(screen_row["count"])
            screen_row["n_mutations"] = mutation_count(screen_row["mutation"])
            out.append(screen_row)
    return out

def load_selectivity():
    out = []
    with open_text(SELECTIVITY_PATH) as f:
        for sel_row in csv.DictReader(f):
            sel_row["ratio"] = float(sel_row["ratio"])
            sel_row["r_ee"] = float(sel_row["r_enantiomeric_excess"])
            sel_row["n_mutations"] = mutation_count(sel_row["mutation"])
            out.append(sel_row)
    return out

screen_rows = load_screen()
selectivity_rows = load_selectivity()
len(screen_rows), len(selectivity_rows)

In [ ]:
def median(values):
    xs = sorted(values)
    n = len(xs)
    if n == 0:
        return None
    mid = n // 2
    if n % 2:
        return xs[mid]
    return (xs[mid - 1] + xs[mid]) / 2

def mean(values):
    return sum(values) / len(values)

def group_by(items, key):
    grouped = {}
    for item in items:
        grouped.setdefault(item[key], []).append(item)
    return grouped

In [ ]:
screen_median = median([r["ratio"] for r in screen_rows])
selectivity_median = median([r["r_ee"] for r in selectivity_rows])
mo.md(
    f"""
    **Dataset snapshot.** The activity screen has **{len(screen_rows):,} variants**.
    The smaller selectivity table has **{len(selectivity_rows):,} variants** with both activity ratio
    and R-enantiomeric excess. The full activity landscape is highly sparse: median ratio is
    **{screen_median:.3f}**, while the selectivity follow-up median R-ee is **{selectivity_median:.1%}**.
    """
)

**Dataset snapshot.** The activity screen has **11,305 variants**.
The smaller selectivity table has **427 variants** with both activity ratio
and R-enantiomeric excess. The full activity landscape is highly sparse: median ratio is
**0.006**, while the selectivity follow-up median R-ee is **41.6%**.

In [ ]:
def summarize_activity(items):
    out = []
    for n_mut, mc_group in sorted(group_by(items, "n_mutations").items()):
        out.append({
            "n_mutations": n_mut,
            "n": len(mc_group),
            "median_ratio": median([r["ratio"] for r in mc_group]),
            "top_ratio": max(r["ratio"] for r in mc_group),
        })
    return out

def summarize_selectivity(items):
    out = []
    for n_mut, mc_group in sorted(group_by(items, "n_mutations").items()):
        out.append({
            "n_mutations": n_mut,
            "n": len(mc_group),
            "median_ratio": median([r["ratio"] for r in mc_group]),
            "median_r_ee": median([r["r_ee"] for r in mc_group]),
        })
    return out

by_count_activity = summarize_activity(screen_rows)
by_count_selectivity = summarize_selectivity(selectivity_rows)
by_count_activity, by_count_selectivity

In [ ]:
mc_fig = go.Figure()
mc_fig.add_trace(go.Bar(
    x=[str(r["n_mutations"]) for r in by_count_selectivity],
    y=[r["median_ratio"] for r in by_count_selectivity],
    name="median activity ratio",
    marker_color="#2563eb",
))
mc_fig.add_trace(go.Scatter(
    x=[str(r["n_mutations"]) for r in by_count_selectivity],
    y=[r["median_r_ee"] for r in by_count_selectivity],
    name="median R-selectivity",
    mode="lines+markers",
    yaxis="y2",
    marker_color="#dc2626",
))
mc_fig.update_layout(
    title="Combinatorial variants dominate both activity and selectivity",
    xaxis_title="Number of mutations in variant",
    yaxis_title="Median activity ratio",
    yaxis2=dict(title="Median R-ee", overlaying="y", side="right", tickformat=".0%"),
    template="plotly_white",
    height=440,
)
mc_fig

<marimo-plotly data-figure='{"data":[{"marker":{"color":"#2563eb"},"name":"median activity ratio","x":["1","2","3","4","5","7"],"y":[0.279139285,0.468631166,0.626247293,0.566647766,0.719084213,0.139918602],"type":"bar"},{"marker":{"color":"#dc2626"},"mode":"lines+markers","name":"median R-selectivity","x":["1","2","3","4","5","7"],"y":[0.328894897,0.49553243,0.47149574699999997,0.978065935,0.981058804,-0.57633244],"yaxis":"y2","type":"scatter"}],"layout":{"template":{"data":{"barpolar":[{"marker":{"line":{"color":"white","width":0.5},"pattern":{"fillmode":"overlay","size":10,"solidity":0.2}},"type":"barpolar"}],"bar":[{"error_x":{"color":"#2a3f5f"},"error_y":{"color":"#2a3f5f"},"marker":{"line":{"color":"white","width":0.5},"pattern":{"fillmode":"overlay","size":10,"solidity":0.2}},"type":"bar"}],"carpet":[{"aaxis":{"endlinecolor":"#2a3f5f","gridcolor":"#C8D4E3","linecolor":"#C8D4E3","minorgridcolor":"#C8D4E3","startlinecolor":"#2a3f5f"},"baxis":{"endlinecolor":"#2a3f5f","gridcolor":"#C8D4E3","linecolor":"#C8D4E3","minorgridcolor":"#C8D4E3","startlinecolor":"#2a3f5f"},"type":"carpet"}],"choropleth":[{"colorbar":{"outlinewidth":0,"ticks":""},"type":"choropleth"}],"contourcarpet":[{"colorbar":{"outlinewidth":0,"ticks":""},"type":"contourcarpet"}],"contour":[{"colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]],"type":"contour"}],"heatmap":[{"colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]],"type":"heatmap"}],"histogram2dcontour":[{"colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]],"type":"histogram2dcontour"}],"histogram2d":[{"colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]],"type":"histogram2d"}],"histogram":[{"marker":{"pattern":{"fillmode":"overlay","size":10,"solidity":0.2}},"type":"histogram"}],"mesh3d":[{"colorbar":{"outlinewidth":0,"ticks":""},"type":"mesh3d"}],"parcoords":[{"line":{"colorbar":{"outlinewidth":0,"ticks":""}},"type":"parcoords"}],"pie":[{"automargin":true,"type":"pie"}],"scatter3d":[{"line":{"colorbar":{"outlinewidth":0,"ticks":""}},"marker":{"colorbar":{"outlinewidth":0,"ticks":""}},"type":"scatter3d"}],"scattercarpet":[{"marker":{"colorbar":{"outlinewidth":0,"ticks":""}},"type":"scattercarpet"}],"scattergeo":[{"marker":{"colorbar":{"outlinewidth":0,"ticks":""}},"type":"scattergeo"}],"scattergl":[{"marker":{"colorbar":{"outlinewidth":0,"ticks":""}},"type":"scattergl"}],"scattermapbox":[{"marker":{"colorbar":{"outlinewidth":0,"ticks":""}},"type":"scattermapbox"}],"scattermap":[{"marker":{"colorbar":{"outlinewidth":0,"ticks":""}},"type":"scattermap"}],"scatterpolargl":[{"marker":{"colorbar":{"outlinewidth":0,"ticks":""}},"type":"scatterpolargl"}],"scatterpolar":[{"marker":{"colorbar":{"outlinewidth":0,"ticks":""}},"type":"scatterpolar"}],"scatter":[{"fillpattern":{"fillmode":"overlay","size":10,"solidity":0.2},"type":"scatter"}],"scatterternary":[{"marker":{"colorbar":{"outlinewidth":0,"tic

**Interpretation.** The transition from single mutants to composed variants is the main story.
In the selectivity table, single mutants have **0.279 median activity** and **32.9% median R-ee**.
Three-mutation variants reach **0.626 median activity**, while four- and five-mutation variants reach
roughly **98% median R-ee**. The best region of this landscape is therefore not "more mutations is always
better"; it is the narrower zone where a few compatible mutations jointly lift activity and stereocontrol.

In [ ]:
def summarize_experiments(items):
    out = []
    for experiment, exp_group in sorted(group_by(items, "experiment").items()):
        out.append({
            "experiment": experiment,
            "n": len(exp_group),
            "median_ratio": median([r["ratio"] for r in exp_group]),
            "median_r_ee": median([r["r_ee"] for r in exp_group]),
            "median_mutations": median([r["n_mutations"] for r in exp_group]),
        })
    return out

by_experiment = summarize_experiments(selectivity_rows)
by_experiment

In [ ]:
exp_fig = go.Figure()
exp_fig.add_trace(go.Scatter(
    x=[r["median_ratio"] for r in by_experiment],
    y=[r["median_r_ee"] for r in by_experiment],
    mode="markers+text",
    text=[r["experiment"] for r in by_experiment],
    textposition="top center",
    marker=dict(
        size=[max(10, r["n"] ** 0.5 * 2.5) for r in by_experiment],
        color=[r["median_mutations"] for r in by_experiment],
        colorscale="Viridis",
        colorbar=dict(title="median mutation count"),
        line=dict(color="white", width=1),
    ),
    hovertemplate=(
        "%{text}<br>median activity=%{x:.3f}<br>"
        "median R-ee=%{y:.1%}<extra></extra>"
    ),
))
exp_fig.update_layout(
    title="Engineering rounds move from sparse hits to active, selective designs",
    xaxis_title="Median activity ratio",
    yaxis_title="Median R-enantiomeric excess",
    yaxis=dict(tickformat=".0%"),
    template="plotly_white",
    height=500,
)
exp_fig

<marimo-plotly data-figure='{"data":[{"hovertemplate":"%{text}<br>median activity=%{x:.3f}<br>median R-ee=%{y:.1%}<extra></extra>","marker":{"color":[1.0,2,2,4.0,2.5,2.0,3,3.0],"colorbar":{"title":{"text":"median mutation count"}},"colorscale":[[0.0,"#440154"],[0.1111111111111111,"#482878"],[0.2222222222222222,"#3e4989"],[0.3333333333333333,"#31688e"],[0.4444444444444444,"#26828e"],[0.5555555555555556,"#1f9e89"],[0.6666666666666666,"#35b779"],[0.7777777777777778,"#6ece58"],[0.8888888888888888,"#b5de2b"],[1.0,"#fde725"]],"line":{"color":"white","width":1},"size":[24.748737341529164,18.200274723201296,15.612494995995995,15.0,10,11.180339887498949,23.84848003542364,23.45207879911715]},"mode":"markers+text","text":["DMS","EPPCR1","EPPCR2","EPPCR3","FragLib","LowN","ML","SGM"],"textposition":"top center","x":[0.298461618,0.106431141,0.683195932,0.7001739979999999,0.7056754605,0.37309765,0.488471387,0.715557132],"y":[0.3129350045,0.458509564,0.975281691,0.982368839,0.9816757705,0.185792388,0.44647827,0.32349935500000004],"type":"scatter"}],"layout":{"template":{"data":{"barpolar":[{"marker":{"line":{"color":"white","width":0.5},"pattern":{"fillmode":"overlay","size":10,"solidity":0.2}},"type":"barpolar"}],"bar":[{"error_x":{"color":"#2a3f5f"},"error_y":{"color":"#2a3f5f"},"marker":{"line":{"color":"white","width":0.5},"pattern":{"fillmode":"overlay","size":10,"solidity":0.2}},"type":"bar"}],"carpet":[{"aaxis":{"endlinecolor":"#2a3f5f","gridcolor":"#C8D4E3","linecolor":"#C8D4E3","minorgridcolor":"#C8D4E3","startlinecolor":"#2a3f5f"},"baxis":{"endlinecolor":"#2a3f5f","gridcolor":"#C8D4E3","linecolor":"#C8D4E3","minorgridcolor":"#C8D4E3","startlinecolor":"#2a3f5f"},"type":"carpet"}],"choropleth":[{"colorbar":{"outlinewidth":0,"ticks":""},"type":"choropleth"}],"contourcarpet":[{"colorbar":{"outlinewidth":0,"ticks":""},"type":"contourcarpet"}],"contour":[{"colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]],"type":"contour"}],"heatmap":[{"colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]],"type":"heatmap"}],"histogram2dcontour":[{"colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]],"type":"histogram2dcontour"}],"histogram2d":[{"colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]],"type":"histogram2d"}],"histogram":[{"marker":{"pattern":{"fillmode":"overlay","size":10,"solidity":0.2}},"type":"histogram"}],"mesh3d":[{"colorbar":{"outlinewidth":0,"ticks":""},"type":"mesh3d"}],"parcoords":[{"line":{"colorbar":{"outlinewidth":0,"ticks":""}},"type":"parcoords"}],"pie":[{"automargin":true,"type":"pie"}],"scatter3d":[{"line":{"colorbar":{"outlinewidth":0,"ticks":""}},"marker":{"colorbar":{"outlinewidth":0,"ticks":""}},"type":"scatter3d"}],"scattercarpet":[{"marker":{"colorbar":{"outlinewidth":0,"ticks":""}},"type":"scattercarpet"}],"scattergeo":[{"marker":{"colorbar":{"outlinewidth":0,"ticks":""}},"type":"scattergeo"}],"scattergl":[{"m

**Round-level finding.** The early DMS rows are broad but weak: **98 DMS variants** have only
**0.298 median activity** and **31.3% median R-ee**. EPPCR2 and EPPCR3 are dramatically different:
**EPPCR2 reaches 97.5% median R-ee with 0.683 median activity**, and **EPPCR3 reaches 98.2% median
R-ee with 0.700 median activity**. That is the campaign's phase change: later rounds do not merely
add mutations, they preserve useful motifs while recombining them into a high-fitness region.

In [ ]:
def collect_high_fitness(items):
    return [
        r for r in items
        if r["ratio"] >= 0.70 and r["r_ee"] >= 0.95
    ]

def count_motifs(items):
    counter = Counter()
    for hf_row in items:
        for mutation in [m.strip() for m in hf_row["mutation"].split(";") if m.strip()]:
            counter[mutation] += 1
    return counter

def top_winners(items, n):
    return sorted(
        items,
        key=lambda r: (r["ratio"], r["r_ee"]),
        reverse=True,
    )[:n]

high_fitness = collect_high_fitness(selectivity_rows)
motif_counter = count_motifs(high_fitness)
top_high_fitness = top_winners(high_fitness, 10)

motif_counter.most_common(10), top_high_fitness

In [ ]:
motif_lines = "\n".join(
    f"- `{mutation}` appears in {count} high-fitness variants"
    for mutation, count in motif_counter.most_common(8)
)
mo.md(
    f"""
    ## The reusable mutation program

    Define a high-fitness hit as **activity ratio >= 0.70** and **R-ee >= 95%**.
    There are **{len(high_fitness)}** such variants. The recurring mutations are:

    {motif_lines}

    **Lift check.** S220T appears in **100% (34/34)** of high-fitness winners but only
    **18.3% (78/427)** of the full selectivity table — a **5.5× enrichment**. That ratio,
    not the raw count, is the central evidence for the motif claim. H230Y is similarly enriched.

    The clearest motif is **S220T**, usually paired with **H230Y** in the strongest EPPCR3 variants.
    This is the actionable part: the model or experimentalist should not rank mutations independently.
    It should search around compatible mutation neighborhoods.
    """
)

    ## The reusable mutation program

    Define a high-fitness hit as **activity ratio >= 0.70** and **R-ee >= 95%**.
    There are **34** such variants. The recurring mutations are:

    - `S220T` appears in 34 high-fitness variants
- `H230Y` appears in 18 high-fitness variants
- `R148P` appears in 5 high-fitness variants
- `R137H` appears in 2 high-fitness variants
- `A169S` appears in 1 high-fitness variants
- `A169T` appears in 1 high-fitness variants
- `A187S` appears in 1 high-fitness variants
- `A202V` appears in 1 high-fitness variants

    **Lift check.** S220T appears in **100% (34/34)** of high-fitness winners but only
    **18.3% (78/427)** of the full selectivity table — a **5.5× enrichment**. That ratio,
    not the raw count, is the central evidence for the motif claim. H230Y is similarly enriched.

    The clearest motif is **S220T**, usually paired with **H230Y** in the strongest EPPCR3 variants.
    This is the actionable part: the model or experimentalist should not rank mutations independently.
    It should search around compatible mutation neighborhoods.
    

In [ ]:
table_rows = "\n".join(
    f"| {r['mutation']} | {r['experiment']} | {r['ratio']:.3f} | {r['r_ee']:.1%} |"
    for r in top_high_fitness[:8]
)
mo.md(
    f"""
    ## Representative winners

    | mutation set | round | activity ratio | R-ee |
    |---|---:|---:|---:|
    {table_rows}
    """
)

    ## Representative winners

    | mutation set | round | activity ratio | R-ee |
    |---|---:|---:|---:|
    | V175A; S220T; H230Y; V232D | EPPCR3 | 0.855 | 98.6% |
| Q194L; S220T; H230Y | EPPCR3 | 0.853 | 99.0% |
| A202V; S220T; H230Y | EPPCR3 | 0.828 | 98.4% |
| V97A; S220T; H230Y | EPPCR3 | 0.812 | 98.5% |
| T81S; A92V; S220T; H230Y | EPPCR3 | 0.809 | 98.3% |
| S205C; S220T; H230Y | EPPCR3 | 0.801 | 98.5% |
| A187S; S220T | EPPCR2 | 0.797 | 97.9% |
| D170E; S220T; H230Y | EPPCR3 | 0.795 | 98.4% |
    

## What this means

**Found:** In this protein fitness landscape, the high-value variants are not explained by single-mutation
screening alone. Fitness emerges when a small number of compatible mutations are composed, especially
around reusable motifs such as `S220T + H230Y`.

**Why it matters:** This is exactly the failure mode that makes protein engineering hard: a mutation can
look unimpressive alone but become valuable in the right background. An AI-guided search policy should
therefore spend less effort on independent single-mutant ranking and more effort on motif-conditioned
combinatorial neighborhoods.

**Caveats:** The notebook uses summary assay tables rather than raw chromatograms, so it cannot audit
measurement noise beyond the provided counts and ratios. It also analyzes one enzyme family, not all
protein fitness landscapes. The claim is intentionally local: this IRED campaign shows a strong
compositional fitness signal.